# PyCAM-SIMA interactive workflow

This is the single maintained PyCAM-SIMA Notebook. It covers three related workflows:

1. run the complete 24-rank CAM-SIMA model one timestep at a time;
2. pause the complete model after each top-level CAM phase and inspect a field;
3. experiment with the small Kessler kernel's Python `StepPlan`, process switches, and live parameter view.

The complete model and the small kernel are separate drivers. The complete-model no-physics experiment uses the `FADIAB` profile; the small kernel switches do not disable physics inside a full FKESSLER CAM run.

## Part 1 — complete CAM-SIMA MPI model

Choose `kessler` for the validated physics case or `adiabatic` for CAM-SIMA's dynamics-only configuration. From a Derecho login-node kernel, `start()` submits the worker through PBS; inside a compute allocation it launches locally.

In [3]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import pycam_sima
from pycam_sima import NotebookSession

repo = Path("/glade/work/ruitong/pycam-sima")
scratch = Path(os.environ.get("SCRATCH", "/glade/derecho/scratch/ruitong"))
physics_profile = "kessler"  # Change to "adiabatic" for dynamics only.

profiles = {
    "kessler": {
        "config": repo / "configs/fkessler_ne3pg3.yaml",
        "case": repo / "reference/cases/FKESSLER_ne3pg3_gnu_24x50",
        "reference_run": scratch / "pycam-sima/FKESSLER_ne3pg3_gnu_24x50/FKESSLER_ne3pg3_gnu_24x50/run",
    },
    "adiabatic": {
        "config": repo / "configs/adiabatic_ne3pg3.yaml",
        "case": repo / "reference/cases/FADIAB_ne3pg3_gnu_24x50",
        "reference_run": scratch / "pycam-sima/FADIAB_ne3pg3_gnu_24x50/FADIAB_ne3pg3_gnu_24x50/run",
    },
}
selected = profiles[physics_profile]
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir = scratch / "pycam-sima/notebook_trials" / f"{physics_profile}-{stamp}" / "run"
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(selected["reference_run"] / "atm_in", run_dir / "atm_in")
print("pycam_sima", pycam_sima.__version__)
print(physics_profile, run_dir)

pycam_sima 0.4.0
kessler /glade/derecho/scratch/ruitong/pycam-sima/notebook_trials/kessler-20260718-220052/run


### Start the MPI worker

This returns only after all 24 ranks have completed initialization and reached the command wait loop.

In [4]:
if "model" in globals() and model.running:
    model.close()

model = NotebookSession(
    selected["config"],
    run_dir=run_dir,
    env_script=selected["case"] / ".env_mach_specific.sh",
    python_executable=repo / ".venv/bin/python",
    log_path=run_dir / "mpi-worker.log",
)
model.start()
print(
    f"ready: mode={model.launch_mode_used}, job={model.job_id}, "
    f"ranks={model.ranks}, fields={len(model.field_names)}, step={model.current_step}"
)
print("next phase:", model.next_phase)

PyCAM-SIMA PBS worker submitted as 6792524.desched1; waiting for 24 MPI ranks ...
ready: mode=pbs, job=6792524.desched1, ranks=24, fields=21, step=0
next phase: cam_run2


### Inspect live fields

`get_field()` returns a rank-local NumPy copy. `get_field_stats()` avoids transferring the whole array.

In [5]:
field_name = "air_temperature"
rank = 0
print(model.field_names)
print(model.field_info(field_name))
print(model.get_field_stats(field_name, rank=rank))
temperature = model.get_field(field_name, rank=rank)
temperature

('air_temperature', 'eastward_wind', 'northward_wind', 'surface_air_pressure', 'air_pressure_thickness', 'air_pressure_thickness_of_dry_air', 'air_pressure', 'air_pressure_of_dry_air', 'air_pressure_at_interface', 'air_pressure_of_dry_air_at_interface', 'surface_pressure_of_dry_air', 'surface_geopotential', 'geopotential_height_wrt_surface', 'geopotential_height_wrt_surface_at_interface', 'lagrangian_tendency_of_air_pressure', 'reciprocal_of_dimensionless_exner_function_wrt_surface_air_pressure', 'dry_static_energy', 'tendency_of_air_temperature_due_to_model_physics', 'tendency_of_eastward_wind_due_to_model_physics', 'tendency_of_northward_wind_due_to_model_physics', 'ccpp_constituents')
{'shape': (27, 30), 'dtype': '<f8', 'dimensions': ('horizontal_dimension', 'vertical_layer_dimension'), 'owner': 'native_view'}
{'rank': 0, 'shape': (27, 30), 'dtype': '<f8', 'min': 149.8407866754426, 'max': 306.29054325059553, 'mean': 237.51537148398492}


array([[150.20253396, 158.94337244, 167.16118088, 174.73367559,
        181.57244239, 187.71430852, 193.05934551, 197.28293981,
        201.03648614, 205.10012764, 209.52062187, 214.33856299,
        219.59632439, 225.33312797, 231.5994718 , 238.41764954,
        245.79321125, 253.68209419, 261.98531497, 269.87826244,
        276.48345972, 281.6156288 , 285.31699146, 287.64103729,
        289.22459305, 290.65616286, 291.93823274, 293.07193045,
        294.05730233, 294.89447155],
       [150.32655508, 159.19389267, 167.58283913, 175.34162418,
        182.34181255, 188.59546082, 193.99676679, 198.23268511,
        201.97220267, 205.99395396, 210.33777076, 215.03799062,
        220.13024923, 225.64641816, 231.62669153, 238.08725527,
        245.02996247, 252.4160406 , 260.16325502, 267.52475172,
        273.70461795, 278.53314474, 282.03501367, 284.24340996,
        285.75205094, 287.11841745, 288.34391967, 289.42887324,
        290.37274642, 291.17523502],
       [150.42552706, 159.3942

### Advance one complete timestep

All MPI ranks execute together and return to the wait loop when the timestep is complete.

In [6]:
step = model.step()
print("completed step:", step)
print(model.get_field_stats(field_name, rank=rank))

completed step: 1
{'rank': 0, 'shape': (27, 30), 'dtype': '<f8', 'min': 149.84036762606746, 'max': 306.30091247707594, 'mean': 237.5147291811104}


## Part 2 — pause after each complete-CAM phase

`run_phase()` sends one command to all 24 ranks. After the phase completes, every rank waits again, so fields can be inspected or changed before the next phase. Re-run the next code cell repeatedly to walk through the model.

In [7]:
print("validated order:", model.phase_names)
print("current status:", model.phase_status)

validated order: ('cam_run2', 'cam_run3', 'cam_run4', 'cam_timestep_final', 'advance_timestep', 'cam_timestep_init', 'cam_run1')
current status: {'last_phase': 'cam_run1', 'next_phase': 'cam_run2', 'sequence_safe': True, 'cycle_kind': 'requested_step', 'cycle_complete': True, 'step': 1, 'native_nstep': 2}


In [8]:
phase = model.next_phase
status = model.run_phase(phase)
stats = model.get_field_stats(field_name, rank=rank)
print(
    f"finished={phase} next={status['next_phase']} "
    f"step={status['step']} native_nstep={status['native_nstep']} "
    f"Tmean={stats['mean']:.17g}"
)

finished=cam_run2 next=cam_run3 step=1 native_nstep=2 Tmean=237.51472918109323


To execute one whole phase cycle automatically while still collecting each boundary, use the following optional cell.

In [9]:
phase_results = []
for _ in model.phase_names:
    phase = model.next_phase
    status = model.run_phase(phase)
    stats = model.get_field_stats(field_name, rank=rank)
    phase_results.append((phase, status["step"], stats["mean"]))
phase_results

[('cam_run3', 1, 237.51472918109323),
 ('cam_run4', 1, 237.51472918109323),
 ('cam_timestep_final', 1, 237.51472918109323),
 ('advance_timestep', 2, 237.51472918109323),
 ('cam_timestep_init', 2, 237.5144615520627),
 ('cam_run1', 2, 237.514461662134),
 ('cam_run2', 2, 237.51446166211662)]

### Optional live edit

This intentionally breaks BFB. Uncomment only for an intervention experiment. The modified rank-zero array is consumed by the next CAM phase.

In [10]:
# changed = model.get_field("air_temperature", rank=0)
# changed[0, 0] += 0.01
# model.set_field("air_temperature", changed, rank=0)
# print(model.get_field_stats("air_temperature", rank=0))

### Optional unsafe ordering experiment

The safe state machine rejects any phase other than `model.next_phase`. An explicitly unsafe call marks the session unsafe and disables normal `step()` sequencing.

In [11]:
print("safe next phase:", model.next_phase)
# model.run_phase("cam_run3")  # Rejected unless cam_run3 is next.
# model.run_phase("cam_run3", allow_unsafe_order=True)

safe next phase: cam_run3


### Finalize the MPI model

Always close the session so CAM finalizes and the MPI worker exits.

In [12]:
model.close()
print("closed:", run_dir)

closed: /glade/derecho/scratch/ruitong/pycam-sima/notebook_trials/kessler-20260718-220052/run


### Optional BFB comparison

This fail-closed comparison is meaningful after running the complete configured 50 steps without live edits or unsafe reordering.

In [13]:
from pycam_sima.history_compare import compare_history

comparison = compare_history(selected["reference_run"], run_dir)
comparison.to_dict()

{'bfb': False,
 'reference_files': 51,
 'candidate_files': 3,
 'compared_files': 3,
 'missing_in_candidate': ('0001-01-01-05400.nc',
  '0001-01-01-07200.nc',
  '0001-01-01-09000.nc',
  '0001-01-01-10800.nc',
  '0001-01-01-12600.nc',
  '0001-01-01-14400.nc',
  '0001-01-01-16200.nc',
  '0001-01-01-18000.nc',
  '0001-01-01-19800.nc',
  '0001-01-01-21600.nc',
  '0001-01-01-23400.nc',
  '0001-01-01-25200.nc',
  '0001-01-01-27000.nc',
  '0001-01-01-28800.nc',
  '0001-01-01-30600.nc',
  '0001-01-01-32400.nc',
  '0001-01-01-34200.nc',
  '0001-01-01-36000.nc',
  '0001-01-01-37800.nc',
  '0001-01-01-39600.nc',
  '0001-01-01-41400.nc',
  '0001-01-01-43200.nc',
  '0001-01-01-45000.nc',
  '0001-01-01-46800.nc',
  '0001-01-01-48600.nc',
  '0001-01-01-50400.nc',
  '0001-01-01-52200.nc',
  '0001-01-01-54000.nc',
  '0001-01-01-55800.nc',
  '0001-01-01-57600.nc',
  '0001-01-01-59400.nc',
  '0001-01-01-61200.nc',
  '0001-01-01-63000.nc',
  '0001-01-01-64800.nc',
  '0001-01-01-66600.nc',
  '0001-01-01-684

## Part 3 — editable Kessler kernel step plan

This is the smaller Python-owned-state driver, not the complete SE model above. It calls the real Kessler wrapper `.so`, but its default `IdentityDynamics` does no numerical dynamics work. It is useful for changing physics order, switches, and parameters directly from Python.

In [14]:
from pycam_sima import FKesslerDriver, RuntimeOptions, StepPlan
from pycam_sima.config import CaseConfig
from pycam_sima.native import NativeKesslerBackend

kernel_config = CaseConfig.from_yaml(repo / "configs/fkessler_ne3pg3.yaml")
kernel_options = RuntimeOptions(
    timestep_seconds=1800,
    physics_before=True,
    physics_after=True,
    dynamics=True,
)
kernel_plan = StepPlan.default()
kernel = FKesslerDriver(
    kernel_config,
    backend=NativeKesslerBackend(kernel_config.native.kessler_library),
    options=kernel_options,
    step_plan=kernel_plan,
)
kernel.allocate_minimal_state(ncol=1)
kernel.parameters.surface_reference_pressure = 100_000.0
kernel.parameters.dycore_energy_adjustment = True
kernel.parameters.constituent_minimum_values = (1.0e-12, 0.0, 0.0)
kernel.initialize()
kernel.step_plan.describe(kernel.options)

[{'order': 1,
  'name': 'kessler_after_coupler',
  'required': False,
  'controlled_by': 'physics_after',
  'plan_enabled': True,
  'enabled': True},
 {'order': 2,
  'name': 'physics_to_dynamics',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 3,
  'name': 'se_dynamics',
  'required': False,
  'controlled_by': 'dynamics',
  'plan_enabled': True,
  'enabled': True},
 {'order': 4,
  'name': 'physics_timestep_final',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 5,
  'name': 'advance_clock',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 6,
  'name': 'dynamics_to_physics',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 7,
  'name': 'physics_timestep_initial',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 8,
  'name': 'kessler_

### Run one kernel step and inspect every internal phase boundary

Observers receive the same live `StatePool` arrays passed to the native wrapper.

In [15]:
kernel_phase_stats = []

def record_kernel_phase(context):
    temperature = context.state["air_temperature"]
    kernel_phase_stats.append(
        (context.task_name, float(temperature.min()), float(temperature.max()))
    )

kernel.observe("phase_end:*", record_kernel_phase, access="readonly")
kernel.step()
kernel_phase_stats

[('kessler_after_coupler', 230.05739084027215, 288.2801089722762),
 ('physics_to_dynamics', 230.05739084027215, 288.2801089722762),
 ('se_dynamics', 230.05739084027215, 288.2801089722762),
 ('physics_timestep_final', 230.05739084027215, 288.2801089722762),
 ('advance_clock', 230.05739084027215, 288.2801089722762),
 ('dynamics_to_physics', 230.05739084027215, 288.2801089722762),
 ('physics_timestep_initial', 230.05739084027215, 288.2801089722762),
 ('kessler_before_coupler', 230.05739084027215, 288.2801089722762)]

### Change parameters and process switches between steps

The first two switches skip both Kessler physics sections. With the default `IdentityDynamics`, leaving `dynamics=True` tests control flow only. Use the full `adiabatic` profile in Part 1 for a real SE dynamics-only run.

In [16]:
kernel.options.physics_before = False
kernel.options.physics_after = False
kernel.options.dynamics = True
kernel.parameters.timestep_seconds = 900
kernel.parameters.surface_reference_pressure = 98_500.0

print(kernel.step_plan.describe(kernel.options))
print(kernel.parameters.describe())
kernel.step()
kernel.pool["air_temperature"]

[{'order': 1, 'name': 'kessler_after_coupler', 'required': False, 'controlled_by': 'physics_after', 'plan_enabled': True, 'enabled': False}, {'order': 2, 'name': 'physics_to_dynamics', 'required': True, 'controlled_by': None, 'plan_enabled': True, 'enabled': True}, {'order': 3, 'name': 'se_dynamics', 'required': False, 'controlled_by': 'dynamics', 'plan_enabled': True, 'enabled': True}, {'order': 4, 'name': 'physics_timestep_final', 'required': True, 'controlled_by': None, 'plan_enabled': True, 'enabled': True}, {'order': 5, 'name': 'advance_clock', 'required': True, 'controlled_by': None, 'plan_enabled': True, 'enabled': True}, {'order': 6, 'name': 'dynamics_to_physics', 'required': True, 'controlled_by': None, 'plan_enabled': True, 'enabled': True}, {'order': 7, 'name': 'physics_timestep_initial', 'required': True, 'controlled_by': None, 'plan_enabled': True, 'enabled': True}, {'order': 8, 'name': 'kessler_before_coupler', 'required': False, 'controlled_by': 'physics_before', 'plan_e

array([[230.05739084, 231.11775524, 232.22299154, 233.37499859,
        234.57575561, 235.82732557, 237.13185875, 238.49159641,
        239.90887465, 241.38612844, 242.92589578, 244.53082206,
        246.20366462, 247.94729744, 249.76471611, 251.65904289,
        253.633532  , 255.69157496, 257.83670581, 260.0726052 ,
        262.40310033, 264.83214761, 267.36373923, 270.00144609,
        272.74615678, 275.58574818, 278.46457662, 281.32187751,
        284.52869594, 288.28010897]])

### Optional kernel ordering experiment

Optional phases can be disabled normally. Any reordering or disabling of a required lifecycle phase needs `unsafe=True`.

In [17]:
kernel.step_plan.disable("se_dynamics")
kernel.step_plan.enable("se_dynamics")

# kernel.step_plan.move(
#     "se_dynamics", before="physics_to_dynamics", unsafe=True
# )
# kernel.step_plan.reset()

kernel.step_plan.describe(kernel.options)

[{'order': 1,
  'name': 'kessler_after_coupler',
  'required': False,
  'controlled_by': 'physics_after',
  'plan_enabled': True,
  'enabled': False},
 {'order': 2,
  'name': 'physics_to_dynamics',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 3,
  'name': 'se_dynamics',
  'required': False,
  'controlled_by': 'dynamics',
  'plan_enabled': True,
  'enabled': True},
 {'order': 4,
  'name': 'physics_timestep_final',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 5,
  'name': 'advance_clock',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 6,
  'name': 'dynamics_to_physics',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 7,
  'name': 'physics_timestep_initial',
  'required': True,
  'controlled_by': None,
  'plan_enabled': True,
  'enabled': True},
 {'order': 8,
  'name': 'kessler

In [18]:
kernel.finalize()
print("kernel finalized at step", kernel.clock.step)

kernel finalized at step 2
